# Load Inventory Minifigs from Rebrickable CSV Download
Downloads `inventory_minifigs.csv.gz` from the Rebrickable bulk download page and loads it into a Spark DataFrame.

In [ ]:
import io
import gzip
import requests
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

DOWNLOAD_URL = "https://cdn.rebrickable.com/media/downloads/inventory_minifigs.csv.gz"

In [ ]:
response = requests.get(DOWNLOAD_URL, timeout=60)
response.raise_for_status()

with gzip.open(io.BytesIO(response.content)) as f:
    df_pd = pd.read_csv(f)

print(f"Downloaded {len(df_pd)} rows")
print(df_pd.dtypes)
df_pd.head()

In [ ]:
schema = StructType([
    StructField("inventory_id", IntegerType(), nullable=False),
    StructField("fig_num",      StringType(),  nullable=False),
    StructField("quantity",     IntegerType(), nullable=False),
])

rows = [
    (
        int(row["inventory_id"]),
        str(row["fig_num"]),
        int(row["quantity"]),
    )
    for _, row in df_pd.iterrows()
]

spark = SparkSession.builder.getOrCreate()
df_inventory_minifigs = spark.createDataFrame(rows, schema=schema)

df_inventory_minifigs.printSchema()
df_inventory_minifigs.display(10, truncate=False)

In [ ]:
from datetime import datetime
from pyspark.sql.functions import current_timestamp, lit

# Generate timestamp components for partitioned path and filename
now = datetime.utcnow()
year  = now.strftime("%Y")
month = now.strftime("%m")
day   = now.strftime("%d")
ts    = now.strftime("%Y%m%d_%H%M%S")

# Add audit columns
df_inventory_minifigs_out = (
    df_inventory_minifigs
    .withColumn("_load_datetime", current_timestamp())
    .withColumn("_record_source", lit("CSV_DOWNLOAD"))
)

# Update the base path below to match your Unity Catalog external volume mount point
EXTERNAL_VOLUME_BASE = "/Volumes/lego_brickbase/bronze/raw_data_volume/inventory_minifigs"
PARTITION_PATH       = f"{EXTERNAL_VOLUME_BASE}/{year}/{month}/{day}/{ts}"
TEMP_PATH            = f"{PARTITION_PATH}/_tmp"
FILE_NAME            = f"raw_inventory_minifigs_{ts}.parquet"
FINAL_PATH           = f"{PARTITION_PATH}/{FILE_NAME}"

# Write as a single Parquet file to a temp staging directory
df_inventory_minifigs_out.coalesce(1) \
    .write \
    .mode("overwrite") \
    .parquet(TEMP_PATH)

# Locate the single part file and rename it to the desired filename
part_files = [f.path for f in dbutils.fs.ls(TEMP_PATH) if f.name.startswith("part-")]
dbutils.fs.mv(part_files[0], FINAL_PATH)
dbutils.fs.rm(TEMP_PATH, recurse=True)

print(f"Inventory minifigs written to: {FINAL_PATH}")

In [ ]:
from delta.tables import DeltaTable

DELTA_TABLE_PATH = "/Volumes/lego_brickbase/bronze/delta_volume/inventory_minifigs"
TABLE_NAME       = "lego_brickbase.bronze.inventory_minifigs"

# Read the Parquet file written in the previous step
df_parquet = spark.read.parquet(FINAL_PATH)

if not DeltaTable.isDeltaTable(spark, DELTA_TABLE_PATH):
    # First load — create the external Delta table and enforce append-only
    df_parquet.write \
        .format("delta") \
        .mode("overwrite") \
        .option("path", DELTA_TABLE_PATH) \
        .saveAsTable(TABLE_NAME)

    spark.sql(f"ALTER TABLE {TABLE_NAME} SET TBLPROPERTIES ('delta.appendOnly' = 'true')")
    print(f"Created Delta table {TABLE_NAME} at {DELTA_TABLE_PATH}")
else:
    # Incremental load — append new records only
    df_parquet.write \
        .format("delta") \
        .mode("append") \
        .save(DELTA_TABLE_PATH)
    print(f"Appended to {TABLE_NAME}")

row_count = spark.table(TABLE_NAME).count()
print(f"Total rows in {TABLE_NAME}: {row_count:,}")